# ЛР 3

## Импорты

In [1]:
!pip install tensorflow

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import LambdaCallback
import random
import sys

I0000 00:00:1780236947.129832   12042 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780236947.164118   12042 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780236948.242070   12042 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Загрузка и предобработка текста

In [3]:
with open('text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

maxlen = 40
step = 3

sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i:i+maxlen])
    next_chars.append(text[i+maxlen])

X = np.zeros((len(sentences), maxlen, vocab_size), dtype=np.bool_)
y = np.zeros((len(sentences), vocab_size), dtype=np.bool_)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_idx[char]] = 1
    y[i, char_to_idx[next_chars[i]]] = 1

## Модель LSTM

In [4]:
model = Sequential([
    LSTM(128, input_shape=(maxlen, vocab_size)),
    Dense(vocab_size, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

E0000 00:00:1780236949.235933   12042 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/abonentvneseti/programming/github/sound_processing_labs/.venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │       110,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 86)             │        11,094 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,174 (473.34 KB)

 Trainable params: 121,174 (473.34 KB)

 Non-trainable params: 0 (0.00 B)

## Обучение

In [5]:
def on_epoch_end(epoch, _):
    start_idx = random.randint(0, len(text) - maxlen - 1)
    seed = text[start_idx:start_idx+maxlen]
    print(f'\n--- Эпоха {epoch+1} ---')
    print(f'Затравка: "{seed}"')
    generated = seed
    for _ in range(200):
        x_pred = np.zeros((1, maxlen, vocab_size))
        for t, char in enumerate(seed):
            x_pred[0, t, char_to_idx[char]] = 1
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = np.argmax(preds)
        next_char = idx_to_char[next_index]
        generated += next_char
        seed = seed[1:] + next_char
    print(f'Сгенерировано: "{generated}"\n')

history = model.fit(
    X, y,
    batch_size=128,
    epochs=20,
    verbose=1,
    callbacks=[LambdaCallback(on_epoch_end=on_epoch_end)]
)

Epoch 1/20
127/130 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 3.6689
--- Эпоха 1 ---
Затравка: "ая Вселенную Бессмыслицы.
Искусственный "
Сгенерировано: "ая Вселенную Бессмыслицы.
Искусственный                                                                                                                                                                                                         "

130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 73ms/step - loss: 3.4208
Epoch 2/20
127/130 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 3.2634
--- Эпоха 2 ---
Затравка: "учение прокладывает маршруты, и сигнал б"
Сгенерировано: "учение прокладывает маршруты, и сигнал б                                                                                                                                                                                                        "

130/130 ━━━━━━━━━━━━━━━━━━━━ 9s 72ms/step - loss: 3.2514
Epoch 3/20
129/130 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 3.1882
--- Эпоха 3 ---
Затравка:

## Финальная генерация и сохранение результата

In [6]:
def generate_text(seed_text, length=400, temperature=0.5):
    generated = seed_text
    seed = seed_text[-maxlen:]
    for _ in range(length):
        x_pred = np.zeros((1, maxlen, vocab_size))
        for t, char in enumerate(seed):
            if char in char_to_idx:
                x_pred[0, t, char_to_idx[char]] = 1
        preds = model.predict(x_pred, verbose=0)[0]
        preds = np.log(preds + 1e-7) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)
        next_index = np.random.choice(len(preds), p=preds)
        next_char = idx_to_char[next_index]
        generated += next_char
        seed = seed[1:] + next_char
    return generated

start_idx = random.randint(0, len(text) - maxlen - 1)
seed = text[start_idx:start_idx+maxlen]
print("Затравка:", seed)

result = generate_text(seed, 1000, 0.6)
print("\nСгенерированный текст (1000 символов):")
print(result)

with open('generated.txt', 'w', encoding='utf-8') as f:
    f.write(result)
print("\nРезультат сохранён в generated.txt")

Затравка: огда мы, люди, смеёмся, а модели записыв

Сгенерированный текст (1000 символов):
огда мы, люди, смеёмся, а модели записывают чаторек ссвепроенте мерие такстовани  спогда вате на иронают абракадабовно полекте на нейсоте прознают вадали ка т кулишь искускают инколовеко нехоне прокаственной ини продали забовок я провоци кистранных инит миль овора вака данных ичтиктент а стой собуют бузака.
И ногов вый кадатвых порорами, ка ных сала к бруда т налодеми. «Сенено налилийе текутьные и кистое т могошь но кучишь см дать ускорсяных мизет я ирогосмие ные сотслина смодне прераскуслив в пой четов о содей закуса на гореко сводель ных ватую сяденные налуча на далдеми, ка додные посети факту, на вой чишь и мировыка смиры абучка дар селующие на мегганат ракатьшь ворое тактуа превесте счи, вогов фуклика «рорека сбудают уна сото са смоде тволута в обрака, онтей на отоль о мобу де на «чете се свойров. Е четони  «бодашь не ирогизаяты нот ликоя, — собо погом миллектуанных сита тват в рогиз вога сят

## Вывод:
1. Реализована посимвольная языковая модель на основе одного LSTM-слоя (128 юнитов) с использованием TensorFlow/Keras.
2. Модель обучена на текстовом файле `text.txt`, содержащем ~30 000 символов, полученных путём пятикратного повторения исходного отрывка об искусственном интеллекте.
3. По окончании обучения (20 эпох) сеть способна генерировать текст, который внешне напоминает русскоязычный, но является абракадаброй. Пример сгенерированного фрагмента приведён в файле `generated.txt`.
4. Основные причины получения именно абракадабры, а не осмысленного текста:
- малый объём и однообразие обучающего корпуса;
- небольшое число эпох обучения;
- посимвольная генерация без учёта слов как целостных единиц.